# CardiLearn Source Rescue Audit — GEO / Expression Atlas / ARCHS4 / recount3

This Colab determines whether the locked CardiLearn benchmark studies can obtain an acceptable count source without reprocessing all SRA reads.

It is evidence discovery, not automatic scientific approval.

The tool:
- reads `data/manifest.lock.json`;
- resolves authoritative GEO sample metadata;
- resolves GEO supplementary candidates;
- resolves GEO/SRA sample links and ENA run accessions;
- searches EBI ArrayExpress/BioStudies and checks Expression Atlas experiment pages;
- optionally checks ARCHS4 sample availability from a local H5;
- checks recount3 project/run availability;
- classifies sources as strict candidates, derived fallbacks, or rejected;
- writes JSON and CSV reports.

The strict benchmark contract remains unchanged: transformed expression and rounded Kallisto pseudocounts are never silently promoted to raw counts.


In [ ]:
from pathlib import Path
import subprocess, sys, json, pandas as pd

REPO = Path("/content/Virelion-CardiLearn")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Virelion-Biotech/Virelion-CardiLearn.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=False)

sys.path.insert(0, str(REPO))
print(subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())


## Configuration

ARCHS4 is optional because the current mouse gene H5 is tens of gigabytes. The auditor only reads its metadata when you explicitly provide the H5 path. ARCHS4 gene-level values are treated as rounded Kallisto-derived pseudocounts, not as strict raw counts.


In [ ]:
MANIFEST = REPO / "data" / "manifest.lock.json"
OUTPUT = Path("/content/source_rescue_v1.json")

# Optional local ARCHS4 mouse H5:
ARCHS4_H5 = None
# Example:
# ARCHS4_H5 = "/content/drive/MyDrive/mouse_gene_v2.5.h5"

print("Manifest:", MANIFEST)
print("Output:", OUTPUT)
print("ARCHS4 H5:", ARCHS4_H5 or "not supplied")


In [ ]:
from scripts.source_rescue_audit import run_audit

payload = run_audit(
    MANIFEST,
    OUTPUT,
    archs4_h5=ARCHS4_H5,
)

print("Audit complete:", OUTPUT)
print("CSV:", OUTPUT.with_suffix(".csv"))


In [ ]:
summary_rows = []
for accession, report in payload["accessions"].items():
    rec = report.get("recommendation", {})
    summary_rows.append({
        "accession": accession,
        "action": rec.get("recommended_action"),
        "source": rec.get("source_type"),
        "srr_count": len(report.get("srr_ids", [])),
        "srx_count": len(report.get("srx_ids", [])),
    })

summary = pd.DataFrame(summary_rows).sort_values("accession")
display(summary)


In [ ]:
rows = pd.read_csv(OUTPUT.with_suffix(".csv"))
display(
    rows[[
        "accession",
        "source_type",
        "status",
        "sample_coverage",
        "sample_expected",
        "candidate_file_count",
        "run_count",
    ]].sort_values(["accession", "source_type"])
)


## Interpreting the output

**validate_strict_candidate** means the auditor found a potentially acceptable raw-count source. It does not approve it. Exact GSM coverage and the actual file contents still need to be validated.

**retain_as_derived_fallback** means recount3 found the SRA material, but the resulting read-count representation is derived rather than the original submitter count table.

**do_not_promote_archs4** means the GSMs are present in ARCHS4 but the current ARCHS4 gene-level data are rounded Kallisto pseudocounts.

**reprocess_sra** means no acceptable existing count source was found and the next path is SRA reprocessing.


In [ ]:
print(json.dumps({
    accession: report.get("recommendation", {})
    for accession, report in payload["accessions"].items()
}, indent=2, sort_keys=True))


## Optional: inspect exact source evidence for one study

Replace `ACCESSION` below with one of the four failed cohorts.


In [ ]:
ACCESSION = "GSE232259"
detail = payload["accessions"][ACCESSION]

print("GEO/SRA relations:")
for x in detail.get("sra_relations", []):
    print(" ", x)

print("\nResolved SRX:", detail.get("srx_ids", []))
print("Resolved SRR:", detail.get("srr_ids", []))

for source in detail.get("sources", []):
    print("\nSOURCE:", source["source_type"])
    print("STATUS:", source["status"])
    print("COVERAGE:", source["exact_sample_coverage"], "/", source["exact_sample_expected"])
    print("NOTES:")
    for note in source["notes"]:
        print(" -", note)
    if source["candidate_files"]:
        print("CANDIDATES:")
        for item in source["candidate_files"][:20]:
            print(" -", item)


## Next step

Use the JSON/CSV report to decide exactly which cohorts need SRA reprocessing. Do not modify the locked benchmark or add a candidate source to Step 3's external-count mapping until the candidate's actual count matrix, sample mapping, and SHA-256 have been independently verified.
